# Logits Alignments

In [ ]:
def default_params(): 
    return {
        'current_model': 'M2',
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/extraction',
            'transformation': 'curated',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'raw_logits_path' : '/workspaces/CodeSmells/datax/code_smells/logits',
        
        'logits_path': '../data/raw_logits',
        'output_path': '../data/aligned_tokens',
        'preprocessed_dataset_dir' : '../datax/code_smells/dataset_preprocessing',
        'cache_dir': '../datax/hugging_face_cache',
        'log_file': '../datax/code_smells/logit_extraction.log', 
        'callbacks_dir' : '../datax/code_smells/callbacks',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import random
import numpy as np
from statistics import mean, median
import os
import torch
import gc
from difflib import SequenceMatcher

In [3]:
from datasets import load_dataset, Dataset

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

2024-10-01 02:37:09.538332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-01 02:37:09.556287: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-01 02:37:09.561666: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-01 02:37:09.575569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['transformation']}"
create_folder(log_file)
log_file += '/align_aggr.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [ ]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Model Loading

In [6]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [7]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

#### Load Dataset

In [ ]:
df_actual_ntp = pd.read_json(f"{params['output_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}/raw_logits.json", index_col=0)

In [9]:
df_actual_ntp.head(2)

,msg_id,line,column,end_line,end_column,code_smell,code,func_name,commit_id,repo,...,nloc,token_counts,n_identifiers,repository,year,input_ids,max_prob,min_prob,actual_prob,loss
0,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[('#', 0.26652711629867554), ('solution', 0.05...","[('–,', 3.87315374217323e-10), ('/***/', 1.165...","[('def', 0.00030555526609532535), ('test', 0.0...",0.625628
1,C0103,3,8,3,9,"B = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[('#', 0.26652711629867554), ('solution', 0.05...","[('–,', 3.87315374217323e-10), ('/***/', 1.165...","[('def', 0.00030555526609532535), ('test', 0.0...",0.625628


#### Token Binding

In [11]:
def find_range_of_indexes(positions, search_range):
    """
    Finds the range of indexes in the positions array where the search_range is fully included.
    
    Args:
    - positions: A list of tuples, where each tuple is (start_position, end_position) (inclusive).
    - search_range: A tuple (start_position, end_position), where start_position is inclusive and end_position is exclusive.
    
    Returns:
    - A tuple (start_index, end_index) representing the range of indexes in the positions array where the search_range is included.
    """
    start, end = search_range
    start_index = -1
    end_index = -1

    for i, (pos_start, pos_end) in enumerate(positions):
        if pos_start <= start <= pos_end:  # Find the start of the range
            start_index = i
        if pos_start <= end - 1 <= pos_end and pos_end>=end:  # Find the end of the range
            end_index = i
            break

    if start_index != -1 and end_index != -1:
        return (start_index, end_index)
    else:
        return None  # If no range is found

In [12]:
def get_substring_positions(code: str, code_smell: str, start):
    """
    Calculate the start and end positions of the substring based on line and column information.

    Parameters:
    text (str): The input string containing multiple lines.
    start (tuple): A tuple of (start_line, start_column) indicating the start position.
    end (tuple): A tuple of (end_line, end_column) indicating the end position.

    Returns:
    tuple: A tuple containing (start_position, end_position) of the substring in the input string.
    """
    lines = code.split('\n')  # Split the string into lines

    # Calculate the character position for the start of the substring
    start_line, start_column = start
    
    try:
        start_position = sum(len(lines[i]) + 1 for i in range(start_line - 1)) + start_column
    except:
        start_position = code.find(code_smell)

    if start_line > len(lines) or start_position>= len(code): 
        start_position = code.find(code_smell)

    if start_position == -1:
        match = SequenceMatcher(None, code, code_smell).find_longest_match()
        start_position= match.a
        end_position = match.a + match.size
    else:
        end_position = start_position + len(code_smell)
        end_position = len(code) if end_position >= len(code) else end_position
    

    return (start_position, end_position)

In [13]:
def find_code_smell_logits(code, code_smell_pos, logits_array, tokenizer):
    indexes_range = find_range_of_indexes(tokenizer.encode_plus(code, return_offsets_mapping=True, add_special_tokens=False)['offset_mapping'], code_smell_pos)
    return eval(logits_array)[indexes_range[0]:indexes_range[1]+1]

In [ ]:
### TODO
df_actual_ntp['code_smell_pos'] = df_actual_ntp.apply(lambda row: get_substring_positions(row['code'], row['code_smell'], (row['line'], row['column'])), axis=1)

In [15]:
df_actual_ntp = df_actual_ntp.drop(['line', 'column', 'end_line', 'end_column'], axis=1)

In [16]:
# Alignments
df_actual_ntp['code_smell_actual_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['actual_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_max_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['max_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_min_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['min_prob'], tokenizer), axis=1)

In [17]:
## Aggregations - median
df_actual_ntp['code_smell_actual_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [18]:
## Aggregations  - mean
df_actual_ntp['code_smell_actual_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [19]:
df_actual_ntp

,msg_id,code_smell,code,func_name,commit_id,repo,path,url,language,n_ast_errors,...,code_smell_pos,code_smell_actual_logits,code_smell_max_logits,code_smell_min_logits,code_smell_actual_prob_median,code_smell_max_prob_median,code_smell_min_prob_median,code_smell_actual_prob_mean,code_smell_max_prob_mean,code_smell_min_prob_mean
0,C0103,"A = [1, 2, 3, 3]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,tests\test_backtrack.py,https://github.com/keon/algorithms.git,Python,0,...,"(47, 63)","[(A, 0.004325982183218002), (=, 0.879888951778...","[("""""", 0.17293989658355713), (=, 0.87988895177...","[(AppCompat, 4.013382334799864e-11), (adelph, ...",0.669525,0.767723,1.083107e-13,0.652978,0.742104,3.412930e-12
1,C0103,"B = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,tests\test_backtrack.py,https://github.com/keon/algorithms.git,Python,0,...,"(72, 88)","[(B, 0.24084141850471497), (=, 0.9841102361679...","[(B, 0.24084141850471497), (=, 0.9841102361679...","[(/******/, 1.6283696613328402e-11), (/******/...",0.714194,0.714194,4.506600e-14,0.658645,0.698602,1.238193e-12
2,C0103,"C = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,tests\test_backtrack.py,https://github.com/keon/algorithms.git,Python,0,...,"(97, 113)","[(C, 0.28388792276382446), (=, 0.9941743016242...","[(C, 0.28388792276382446), (=, 0.9941743016242...","[(/******/, 1.2150841964542192e-11), (/******/...",0.857728,0.857728,3.812279e-14,0.729910,0.742337,9.874202e-13
3,C0103,"A = [1, 2, 3, 3]",def test_unique_array_sum_combinations(self):\...,test_unique_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,tests\test_backtrack.py,https://github.com/keon/algorithms.git,Python,0,...,"(54, 70)","[(A, 0.0039259945042431355), (=, 0.89760905504...","[("""""", 0.17227958142757416), (=, 0.89760905504...","[(locale, 3.712284299406399e-11), (adelph, 4.2...",0.684670,0.721483,4.043375e-14,0.647446,0.707749,2.972741e-12
4,C0103,"B = [2, 3, 3, 4]",def test_unique_array_sum_combinations(self):\...,test_unique_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,tests\test_backtrack.py,https://github.com/keon/algorithms.git,Python,0,...,"(79, 95)","[(B, 0.2197747677564621), (=, 0.98485273122787...","[(B, 0.2197747677564621), (=, 0.98485273122787...","[(locale, 1.3876715748706303e-11), (/******/, ...",0.768858,0.768858,4.154428e-14,0.676743,0.702877,1.072995e-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77530,R0913,"def _call_api(self, path, display_id, note='Do...","def _call_api(self, path, display_id, note='Do...",_call_api,0d067e77c3f5527946fb0c22ee1c7011994cba40,yt-dlp,yt_dlp\extractor\dangalplay.py,https://github.com/yt-dlp/yt-dlp.git,Python,0,...,"(0, 94)","[(def, 0.00030555506236851215), (_, 0.00393252...","[(#, 0.2665270268917084), (solution, 0.0588445...","[(–,, 3.873182052860358e-10), (/***/, 1.165266...",0.036839,0.555497,1.551242e-12,0.290641,0.452764,3.038151e-11
77531,C0103,"for ep in traverse_obj(data, ('data', 'items',...","def _entries(self, subcategories, series_slug)...",_entries,0d067e77c3f5527946fb0c22ee1c7011994cba40,yt-dlp,yt_dlp\extractor\dangalplay.py,https://github.com/yt-dlp/yt-dlp.git,Python,0,...,"(437, 516)","[(ep, 0.01947537623345852), (in, 0.76769471168...","[(episode, 0.31299301981925964), (in, 0.767694...","[(unregister, 4.377263742444404e-12), (bbra, 3...",0.491553,0.633138,1.453245e-12,0.489271,0.644034,5.459768e-12
77532,C0103,"qs = {k: v[-1] for k, v in parse_qs(url).items...","def _real_extract(self, url, html=None):\n ...",_real_extract,a4da9db87b6486b270c15dfa07ab5bfedc83f6bd,yt-dlp,yt_dlp\extractor\youporn.py,https://github.com/yt-dlp/yt-dlp.git,Python,0,...,"(186, 240)","[(q, 0.003225734457373619), (s,

#### SAVE

In [20]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [21]:
create_folder(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'])
df_actual_ntp.to_csv(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'aligned_smells.csv')

In [22]:
torch.cuda.empty_cache()
gc.collect()

0